In [0]:
%pip install --upgrade duckdb "pyiceberg[s3fs]" typing_extensions "pyarrow>=16.0.0"

In [ ]:
%pip install --upgrade duckdb "pyiceberg[s3fs]" typing_extensions "pyarrow>=16.0.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
from pyiceberg.catalog import load_catalog
from pyiceberg import __version__
import pyarrow.types
print("Version of pyiceberg: " + __version__)

# Compatibility shim: PyIceberg 0.11.1 requires pa.types.is_string_view (PyArrow 16+)
if not hasattr(pyarrow.types, 'is_string_view'):
    pyarrow.types.is_string_view = lambda t: False

# 1. Fetch credentials
polaris_oauth_client_id = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_client_id")
polaris_oauth_client_secret = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_client_secret")
polaris_oauth_token_url = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_token_url")
polaris_oauth_scope = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_scope")
polaris_base_url = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_base_url")
polaris_warehouse = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_warehouse")

aws_access_key = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="aws_access_key")
aws_secret_key = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="aws_secret_key")
aws_region = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="aws_region")

# 2. Connect to Polaris Iceberg catalog
catalog = load_catalog(
    "polaris",
    **{
        "type": "rest",
        "uri": polaris_base_url,
        "oauth.token.url": polaris_oauth_token_url,
        "credential": f"{polaris_oauth_client_id}:{polaris_oauth_client_secret}",
        "scope": polaris_oauth_scope,
        "warehouse": polaris_warehouse,
        "s3.access-key-id": aws_access_key,
        "s3.secret-access-key": aws_secret_key,
        "s3.region": aws_region,
        "py-io-impl": "pyiceberg.io.fsspec.FsspecFileIO"
    }
)

catalog.list_namespaces()
catalog._session.headers.pop("X-Iceberg-Access-Delegation", None)

print("Polaris catalog connected successfully.")

In [ ]:
# 3. Discover all namespaces and tables in the catalog
all_tables = []
for ns in catalog.list_namespaces():
    ns_name = ns[0] if isinstance(ns, tuple) else ns
    all_tables.extend(catalog.list_tables(ns_name))

print(f"Found {len(all_tables)} table(s): {[f'{t[0]}.{t[1]}' for t in all_tables]}")

In [ ]:
# 4. Incremental import using DuckDB exclusively for reading Iceberg data
import duckdb
import json
import time
import numpy as np
import pandas as pd_lib
from decimal import Decimal
from datetime import datetime, timezone, date
from pyspark.sql.functions import monotonically_increasing_id, lit
from pyspark.sql.types import StringType, LongType
from pyiceberg.types import TimestamptzType

# Initialize DuckDB
con = duckdb.connect()
con.execute("INSTALL iceberg; LOAD iceberg")
con.execute(f"SET s3_access_key_id='{aws_access_key}'")
con.execute(f"SET s3_secret_access_key='{aws_secret_key}'")
con.execute(f"SET s3_region='{aws_region}'")
print("DuckDB initialized for Iceberg reads.")

# Ensure snapshot tracking table exists
spark.sql("""
    CREATE TABLE IF NOT EXISTS _iceberg_snapshot_tracking (
        source_table STRING, snapshot_id STRING, sequence_number BIGINT, updated_at TIMESTAMP
    ) USING DELTA
""")
print("Snapshot tracking table ready.")

run_end_time = datetime.now(timezone.utc)
print(f"Run end time (upper bound): {run_end_time.isoformat()}")


def safe_json_dumps(x):
    """Safely serialize to JSON, handling special types."""
    if x is None:
        return None
    try:
        if pd_lib.isna(x):
            return None
    except (ValueError, TypeError):
        pass
    if isinstance(x, np.ndarray):
        x = x.tolist()
    return json.dumps(x, default=lambda o:
        float(o) if isinstance(o, (Decimal, np.floating)) else
        int(o) if isinstance(o, np.integer) else
        o.isoformat() if isinstance(o, (datetime, date)) else
        o.tolist() if isinstance(o, np.ndarray) else str(o)
    )


def flatten_pandas_df(pdf):
    """Expand dict columns into flat columns; serialize lists/complex types to JSON."""
    cols_to_drop = []
    new_cols = {}

    for col_name in list(pdf.columns):
        if pdf[col_name].dtype != object:
            continue
        sample = pdf[col_name].dropna().head(10)
        if len(sample) == 0:
            continue
        first_val = sample.iloc[0]

        if isinstance(first_val, dict):
            keys_set = set()
            for val in sample:
                if isinstance(val, dict):
                    keys_set.update(val.keys())
            if len(keys_set) <= 20:
                cols_to_drop.append(col_name)
                for key in sorted(keys_set):
                    series = pdf[col_name].apply(lambda x, k=key: x.get(k) if isinstance(x, dict) else None)
                    if series.apply(lambda x: isinstance(x, Decimal)).any():
                        series = series.apply(lambda x: float(x) if isinstance(x, Decimal) else x)
                    new_cols[f"{col_name}_{key}"] = series
            else:
                pdf[col_name] = pdf[col_name].apply(safe_json_dumps)
        elif isinstance(first_val, (list, np.ndarray)):
            pdf[col_name] = pdf[col_name].apply(safe_json_dumps)
        elif isinstance(first_val, (datetime, date)):
            pdf[col_name] = pdf[col_name].apply(lambda x: x.isoformat() if isinstance(x, (datetime, date)) else x)

    if cols_to_drop:
        pdf = pdf.drop(columns=cols_to_drop)
    for name, series in new_cols.items():
        sample = series.dropna().head(5)
        if len(sample) > 0:
            fv = sample.iloc[0]
            if isinstance(fv, (list, np.ndarray)):
                series = series.apply(safe_json_dumps)
            elif isinstance(fv, (datetime, date)):
                series = series.apply(lambda x: x.isoformat() if isinstance(x, (datetime, date)) else x)
        pdf[name] = series
    return pdf


def _time_filter(time_col, is_tz, watermark, end_time):
    """Build DuckDB WHERE clause for time-based filtering."""
    end_str = end_time.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
    cast = "TIMESTAMPTZ" if is_tz else "TIMESTAMP"
    sfx = "+00:00" if is_tz else ""
    return f'WHERE "{time_col}" > \'{watermark}{sfx}\'::{cast} AND "{time_col}" <= \'{end_str}{sfx}\'::{cast}'


def read_iceberg(metadata_loc, time_col=None, is_tz=False, watermark=None, end_time=None):
    """Read Iceberg table via DuckDB iceberg_scan with optional time filter."""
    where = _time_filter(time_col, is_tz, watermark, end_time) if (time_col and watermark and end_time) else ""
    return con.execute(f"SELECT * FROM iceberg_scan('{metadata_loc}') {where}").fetchdf()


def read_new_files(files, cols=None, time_col=None, is_tz=False, watermark=None, end_time=None):
    """Read new Parquet files directly with optional time filter."""
    if not files:
        return pd_lib.DataFrame()
    file_list = ", ".join(f"'{f}'" for f in files)
    col_expr = ", ".join(f'"{ c}"' for c in cols) if cols else "*"
    where = _time_filter(time_col, is_tz, watermark, end_time) if (time_col and watermark and end_time) else ""
    return con.execute(f"SELECT {col_expr} FROM read_parquet([{file_list}]) {where}").fetchdf()


def get_new_data_files(metadata_loc, seq_num):
    """Get S3 paths of data files added since the stored sequence number."""
    return con.execute(f"""
        SELECT file_path FROM iceberg_metadata('{metadata_loc}')
        WHERE manifest_sequence_number > {seq_num} AND manifest_content = 'DATA' AND status = 'ADDED'
    """).fetchdf()['file_path'].tolist()


def get_watermark(target, time_col):
    """Get max timestamp from existing Delta table."""
    try:
        max_ts = spark.sql(f"SELECT MAX(`{time_col}`) as max_ts FROM {target}").collect()[0]["max_ts"]
        return max_ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] if max_ts else None
    except Exception:
        return None


def get_stored_snapshot(source):
    """Retrieve last-imported snapshot ID and sequence number."""
    try:
        row = spark.sql(f"SELECT snapshot_id, sequence_number FROM _iceberg_snapshot_tracking WHERE source_table = '{source}'").collect()
        return (row[0]["snapshot_id"], row[0]["sequence_number"]) if row else (None, None)
    except Exception:
        return (None, None)


def store_snapshot(source, snapshot_id, seq_num=None):
    """Persist snapshot ID and sequence number."""
    seq_val = seq_num if seq_num is not None else "NULL"
    spark.sql(f"""
        MERGE INTO _iceberg_snapshot_tracking AS t
        USING (SELECT '{source}' AS source_table, '{snapshot_id}' AS snapshot_id, {seq_val} AS sequence_number, current_timestamp() AS updated_at) AS s
        ON t.source_table = s.source_table
        WHEN MATCHED THEN UPDATE SET t.snapshot_id = s.snapshot_id, t.sequence_number = s.sequence_number, t.updated_at = s.updated_at
        WHEN NOT MATCHED THEN INSERT *
    """)


results = []

for ns_name, table_name in all_tables:
    full_path = f"{ns_name}.{table_name}"
    target_table_name = full_path.replace(".", "_")

    print(f"\n{'='*60}")
    print(f"Processing: {full_path} -> {target_table_name}")
    print(f"{'='*60}")

    try:
        t0 = time.time()
        iceberg_table = catalog.load_table(full_path)
        metadata_location = iceberg_table.metadata_location

        # Find time column
        time_col_name, is_tz = None, False
        for field in iceberg_table.schema().fields:
            if field.name in ("time", "lastUpdated", "creationTime"):
                time_col_name, is_tz = field.name, isinstance(field.field_type, TimestamptzType)
                break

        watermark = get_watermark(target_table_name, time_col_name) if time_col_name else None

        # Snapshot-based short-circuit
        snap = iceberg_table.current_snapshot()
        snap_id = str(snap.snapshot_id) if snap else None
        snap_seq = snap.sequence_number if snap else None
        stored_id, stored_seq = get_stored_snapshot(target_table_name)

        if snap_id and snap_id == stored_id:
            print(f"  SKIPPED: Snapshot unchanged ({snap_id}). No S3 scan needed. ({time.time()-t0:.1f}s)")
            results.append({"table": full_path, "status": "skipped", "reason": "snapshot unchanged", "rows": 0})
            continue

        if watermark:
            print(f"  Watermark (last imported): {watermark}")
            print(f"  Fetching data from {watermark} to {run_end_time.isoformat()}")
        elif time_col_name:
            print(f"  No existing data \u2014 full initial load")
        else:
            print(f"  No time column \u2014 full load (table will be overwritten)")

        # Read data
        iceberg_cols = [f.name for f in iceberg_table.schema().fields]

        if watermark and stored_seq is not None:
            try:
                delta_files = get_new_data_files(metadata_location, stored_seq)
                if delta_files:
                    print(f"  Snapshot-diff: {len(delta_files)} new file(s) to read")
                    pdf = read_new_files(delta_files, iceberg_cols, time_col_name, is_tz, watermark, run_end_time)
                else:
                    pdf = pd_lib.DataFrame()
            except Exception as e:
                print(f"  Snapshot-diff failed ({str(e)[:80]}), falling back to iceberg_scan")
                pdf = read_iceberg(metadata_location, time_col_name, is_tz, watermark, run_end_time)
        elif watermark:
            pdf = read_iceberg(metadata_location, time_col_name, is_tz, watermark, run_end_time)
        else:
            pdf = read_iceberg(metadata_location)

        row_count = len(pdf)

        if row_count == 0:
            if snap_id:
                store_snapshot(target_table_name, snap_id, snap_seq)
            print(f"  SKIPPED: No new data since last import. ({time.time()-t0:.1f}s)")
            results.append({"table": full_path, "status": "skipped", "reason": "no new data", "rows": 0})
            continue

        # Flatten and prepare
        pdf = flatten_pandas_df(pdf)
        for c in list(pdf.columns):
            if pdf[c].isna().all():
                pdf[c] = pdf[c].astype("object")

        df = spark.createDataFrame(pdf)

        # Build all column transforms at once (single withColumns call, 1 Analyze RPC)
        schema_fields = df.schema.fields
        null_cols = {f.name for f in schema_fields if str(f.dataType) == "NullType()"}
        transforms = {}
        if null_cols:
            existing_schema = {}
            try:
                existing_schema = {f.name: f.dataType for f in spark.table(target_table_name).schema.fields}
            except Exception:
                pass
            transforms.update({name: lit(None).cast(existing_schema.get(name, StringType())) for name in null_cols})

        try:
            id_offset = spark.sql(f"SELECT COALESCE(MAX(row_id), -1) as max_id FROM {target_table_name}").collect()[0]["max_id"] + 1
        except Exception:
            id_offset = 0
        transforms["row_id"] = monotonically_increasing_id() + lit(id_offset).cast(LongType())
        df = df.withColumns(transforms)

        # Write
        write_mode = "append" if time_col_name else "overwrite"
        schema_opt = "mergeSchema" if time_col_name else "overwriteSchema"
        df.write.mode(write_mode).option(schema_opt, "true").saveAsTable(target_table_name)

        elapsed = time.time() - t0
        if snap_id:
            store_snapshot(target_table_name, snap_id, snap_seq)

        print(f"  SUCCESS: {row_count} rows {write_mode}ed to '{target_table_name}' ({elapsed:.1f}s, duckdb)")
        print(f"  Schema: {[f.name for f in schema_fields] + ['row_id']}")
        results.append({"table": full_path, "status": "success", "rows": row_count, "target": target_table_name, "reader": "duckdb", "mode": write_mode, "seconds": round(elapsed, 1)})

    except Exception as e:
        print(f"  ERROR: {str(e)[:300]}")
        results.append({"table": full_path, "status": "error", "reason": str(e)[:300], "rows": 0})

con.close()
print(f"\nDone. Processed {len(results)} tables.")

In [ ]:
# 5. Summary of all imports
import pandas as pd

df_summary = pd.DataFrame(results)
print(f"\n{'='*60}")
print("IMPORT SUMMARY")
print(f"{'='*60}")
print(f"Total tables processed: {len(results)}")
print(f"Successful: {len(df_summary[df_summary['status'] == 'success'])}")
print(f"Skipped (no new data): {len(df_summary[df_summary['status'] == 'skipped'])}")
print(f"Errors: {len(df_summary[df_summary['status'] == 'error'])}")
print(f"\nTotal new rows imported: {df_summary[df_summary['status'] == 'success']['rows'].sum()}")
print()
display(df_summary)

In [ ]:
# 6. Ensure primary key constraints exist on all imported tables
pk_results = []

for row in results:
    if row["status"] != "success":
        continue
    
    table_name = row["target"]
    constraint_name = f"pk_{table_name}"
    
    try:
        spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN row_id SET NOT NULL")
        spark.sql(f"ALTER TABLE {table_name} ADD CONSTRAINT {constraint_name} PRIMARY KEY (row_id)")
        
        print(f"  \u2713 {table_name}: PRIMARY KEY (row_id) added")
        pk_results.append({"table": table_name, "status": "success"})
    except Exception as e:
        error_msg = str(e)[:300]
        if "already_exist" in error_msg.lower() or "already exists" in error_msg.lower():
            print(f"  \u2713 {table_name}: PRIMARY KEY already exists")
            pk_results.append({"table": table_name, "status": "exists"})
        else:
            print(f"  \u2717 {table_name}: {error_msg}")
            pk_results.append({"table": table_name, "status": "error"})

print(f"\nPrimary keys ensured: {len([r for r in pk_results if r['status'] in ('success', 'exists')])} / {len(pk_results)}")

In [0]:
dbutils.library.restartPython()

In [0]:
from pyiceberg.catalog import load_catalog
from pyiceberg import __version__
import pyarrow.types
print("Version of pyiceberg: " + __version__)

# Compatibility shim: PyIceberg 0.11.1 requires pa.types.is_string_view (PyArrow 16+)
if not hasattr(pyarrow.types, 'is_string_view'):
    pyarrow.types.is_string_view = lambda t: False

# 1. Fetch credentials
polaris_oauth_client_id = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_client_id")
polaris_oauth_client_secret = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_client_secret")
polaris_oauth_token_url = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_token_url")
polaris_oauth_scope = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_oauth_scope")
polaris_base_url = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_base_url")
polaris_warehouse = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="polaris_warehouse")

aws_access_key = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="aws_access_key")
aws_secret_key = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="aws_secret_key")
aws_region = dbutils.secrets.get(scope="CumulocityPolarisIcebergS3", key="aws_region")

# 2. Connect to Polaris Iceberg catalog
catalog = load_catalog(
    "polaris",
    **{
        "type": "rest",
        "uri": polaris_base_url,
        "oauth.token.url": polaris_oauth_token_url,
        "credential": f"{polaris_oauth_client_id}:{polaris_oauth_client_secret}",
        "scope": polaris_oauth_scope,
        "warehouse": polaris_warehouse,
        "s3.access-key-id": aws_access_key,
        "s3.secret-access-key": aws_secret_key,
        "s3.region": aws_region,
        "py-io-impl": "pyiceberg.io.fsspec.FsspecFileIO"
    }
)

catalog.list_namespaces()
catalog._session.headers.pop("X-Iceberg-Access-Delegation", None)

print("Polaris catalog connected successfully.")

In [0]:
# 3. Discover all namespaces and tables in the catalog
all_tables = []

namespaces = catalog.list_namespaces()
print(f"Found {len(namespaces)} namespace(s): {namespaces}")

for ns in namespaces:
    ns_name = ns[0] if isinstance(ns, tuple) else ns
    tables = catalog.list_tables(ns_name)
    for tbl in tables:
        all_tables.append(tbl)
        print(f"  Found table: {tbl[0]}.{tbl[1]}")

print(f"\nTotal tables to import: {len(all_tables)}")

In [0]:
# 4. Incremental import using DuckDB exclusively for reading Iceberg data
import duckdb
import json
import time
import uuid
import numpy as np
import pandas as pd_lib
from decimal import Decimal
from datetime import datetime, timezone, date
from pyspark.sql.functions import monotonically_increasing_id, col, lit, current_timestamp
from pyspark.sql.types import StringType, LongType
from pyiceberg.types import TimestampType, TimestamptzType

# Initialize DuckDB
con = duckdb.connect()
con.execute("INSTALL iceberg")
con.execute("LOAD iceberg")
con.execute(f"SET s3_access_key_id='{aws_access_key}'")
con.execute(f"SET s3_secret_access_key='{aws_secret_key}'")
con.execute(f"SET s3_region='{aws_region}'")
print("DuckDB initialized for Iceberg reads.")

# Current time as upper bound for this run
run_end_time = datetime.now(timezone.utc)
print(f"Run end time (upper bound): {run_end_time.isoformat()}")


class DecimalEncoder(json.JSONEncoder):
    """JSON encoder that handles Decimal and datetime objects."""
    def default(self, obj):
        if isinstance(obj, Decimal):
            return float(obj)
        if isinstance(obj, datetime):
            return obj.isoformat()
        if isinstance(obj, date):
            return obj.isoformat()
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


def safe_json_dumps(x):
    """Safely serialize to JSON, handling pd.NA, NaN, and None."""
    if x is None:
        return None
    try:
        if pd_lib.isna(x):
            return None
    except (ValueError, TypeError):
        pass
    if isinstance(x, np.ndarray):
        return json.dumps(x.tolist(), cls=DecimalEncoder)
    return json.dumps(x, cls=DecimalEncoder)


def flatten_pandas_df(pdf):
    """
    Expand dict columns (from Arrow structs/maps) into flat columns.
    Structs with <=20 keys are expanded; others and lists become JSON strings.
    """
    cols_to_drop = []
    new_cols = {}

    for col_name in pdf.columns:
        if pdf[col_name].dtype == object:
            sample = pdf[col_name].dropna().head(10)
            if len(sample) == 0:
                continue

            first_val = sample.iloc[0]

            if isinstance(first_val, dict):
                keys_set = set()
                for val in sample:
                    if isinstance(val, dict):
                        keys_set.update(val.keys())

                if len(keys_set) <= 20:
                    cols_to_drop.append(col_name)
                    for key in sorted(keys_set):
                        flat_col_name = f"{col_name}_{key}"
                        new_cols[flat_col_name] = pdf[col_name].apply(
                            lambda x, k=key: x.get(k) if isinstance(x, dict) else None
                        )
                        if new_cols[flat_col_name].apply(
                            lambda x: isinstance(x, Decimal)
                        ).any():
                            new_cols[flat_col_name] = new_cols[flat_col_name].apply(
                                lambda x: float(x) if isinstance(x, Decimal) else x
                            )
                else:
                    pdf[col_name] = pdf[col_name].apply(safe_json_dumps)

            elif isinstance(first_val, (list, np.ndarray)):
                pdf[col_name] = pdf[col_name].apply(safe_json_dumps)

    if cols_to_drop:
        pdf = pdf.drop(columns=cols_to_drop)
        for new_col_name, series in new_cols.items():
            pdf[new_col_name] = series

    # Second pass: catch any remaining array-like or datetime columns
    for col_name in pdf.columns:
        if pdf[col_name].dtype == object:
            sample = pdf[col_name].dropna().head(5)
            if len(sample) > 0:
                first_val = sample.iloc[0]
                if isinstance(first_val, (list, np.ndarray)):
                    pdf[col_name] = pdf[col_name].apply(safe_json_dumps)
                elif isinstance(first_val, (datetime, date)):
                    pdf[col_name] = pdf[col_name].apply(
                        lambda x: x.isoformat() if isinstance(x, (datetime, date)) else x
                    )

    return pdf


def get_target_schema(target_table_name):
    """
    Get the schema of the existing Delta table as a dict: {col_name: dataType}.
    Returns None if the table doesn't exist.
    """
    try:
        target_df = spark.table(target_table_name)
        return {field.name: field.dataType for field in target_df.schema.fields}
    except Exception:
        return None


def get_watermark(target_table_name, time_col_name):
    """
    Get the max timestamp (watermark) from the existing Delta table.
    Returns the watermark as ISO string, or None if the table doesn't exist or is empty.
    """
    try:
        result = spark.sql(f"SELECT MAX(`{time_col_name}`) as max_ts FROM {target_table_name}")
        max_ts = result.collect()[0]["max_ts"]
        if max_ts is not None:
            return max_ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
        return None
    except Exception:
        return None


def find_time_column(iceberg_schema):
    """Find the time column from the Iceberg schema."""
    for field in iceberg_schema.fields:
        if field.name in ["time", "lastUpdated", "creationTime"]:
            is_tz = isinstance(field.field_type, TimestamptzType)
            return field.name, is_tz
    return None, False


def read_with_duckdb(metadata_location, time_col_name=None, is_tz=False, watermark=None, end_time=None):
    """Read Iceberg table data using DuckDB with optional time-based filtering."""
    if time_col_name and watermark and end_time:
        end_str = end_time.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]
        if is_tz:
            wm_ts = f"{watermark}+00:00"
            end_ts = f"{end_str}+00:00"
        else:
            wm_ts = watermark
            end_ts = end_str
        query = f"""
            SELECT * FROM iceberg_scan('{metadata_location}')
            WHERE \"{time_col_name}\" > '{wm_ts}'::TIMESTAMP
              AND \"{time_col_name}\" <= '{end_ts}'::TIMESTAMP
        """
    else:
        query = f"SELECT * FROM iceberg_scan('{metadata_location}')"

    duck_result = con.execute(query)
    return duck_result.fetchdf()


results = []

for ns_name, table_name in all_tables:
    full_path = f"{ns_name}.{table_name}"
    target_table_name = full_path.replace(".", "_")

    print(f"\n{'='*60}")
    print(f"Processing: {full_path} -> {target_table_name}")
    print(f"{'='*60}")

    try:
        t0 = time.time()
        iceberg_table = catalog.load_table(full_path)
        metadata_location = iceberg_table.metadata_location

        # Find time column
        time_col_name, is_tz = find_time_column(iceberg_table.schema())

        # Get watermark from existing Delta table
        watermark = None
        if time_col_name:
            watermark = get_watermark(target_table_name, time_col_name)

        if watermark:
            print(f"  Watermark (last imported): {watermark}")
            print(f"  Fetching data from {watermark} to {run_end_time.isoformat()}")
        elif time_col_name:
            print(f"  No existing data \u2014 full initial load")
        else:
            print(f"  No time column \u2014 full load (table will be overwritten)")

        # Read data using DuckDB
        if watermark:
            pdf = read_with_duckdb(metadata_location, time_col_name, is_tz, watermark, run_end_time)
        elif time_col_name:
            # Full load (no watermark yet) - read everything
            pdf = read_with_duckdb(metadata_location)
        else:
            # No time column - full load
            pdf = read_with_duckdb(metadata_location)

        row_count = len(pdf)
        elapsed = time.time() - t0

        if row_count == 0:
            print(f"  SKIPPED: No new data since last import. ({elapsed:.1f}s)")
            results.append({"table": full_path, "status": "skipped", "reason": "no new data", "rows": 0})
            continue

        # Flatten struct/dict columns and serialize lists to JSON strings
        pdf = flatten_pandas_df(pdf)

        # Fix: cast all-null columns to string to avoid void type
        for c in pdf.columns:
            if pdf[c].isna().all():
                pdf[c] = pdf[c].astype("object")

        # Convert to Spark DataFrame
        df = spark.createDataFrame(pdf)

        # Fix: cast NullType columns to match existing target table schema
        existing_schema = get_target_schema(target_table_name)
        for field in df.schema.fields:
            if str(field.dataType) == "NullType()":
                if existing_schema and field.name in existing_schema:
                    target_type = existing_schema[field.name]
                    df = df.withColumn(field.name, lit(None).cast(target_type))
                else:
                    df = df.withColumn(field.name, lit(None).cast(StringType()))

        # Generate unique row_id using offset from existing max
        try:
            max_id_row = spark.sql(f"SELECT COALESCE(MAX(row_id), -1) as max_id FROM {target_table_name}").collect()
            id_offset = max_id_row[0]["max_id"] + 1
        except Exception:
            id_offset = 0

        df = df.withColumn("row_id", monotonically_increasing_id() + lit(id_offset).cast(LongType()))

        # Write mode depends on whether table has a time column (incremental) or not (full replace)
        if time_col_name:
            df.write.mode("append").option("mergeSchema", "true").saveAsTable(target_table_name)
            write_mode = "append"
        else:
            df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table_name)
            write_mode = "overwrite"

        elapsed = time.time() - t0
        print(f"  SUCCESS: {row_count} rows {write_mode}ed to '{target_table_name}' ({elapsed:.1f}s, duckdb)")
        print(f"  Schema: {[f.name for f in df.schema.fields]}")
        results.append({"table": full_path, "status": "success", "rows": row_count, "target": target_table_name, "reader": "duckdb", "mode": write_mode, "seconds": round(elapsed, 1)})

    except Exception as e:
        error_msg = str(e)[:300]
        print(f"  ERROR: {error_msg}")
        results.append({"table": full_path, "status": "error", "reason": error_msg, "rows": 0})

con.close()
print(f"\nDone. Processed {len(results)} tables.")

In [0]:
# 5. Summary of all imports
import pandas as pd

df_summary = pd.DataFrame(results)
print(f"\n{'='*60}")
print("IMPORT SUMMARY")
print(f"{'='*60}")
print(f"Total tables processed: {len(results)}")
print(f"Successful: {len(df_summary[df_summary['status'] == 'success'])}")
print(f"Skipped (no new data): {len(df_summary[df_summary['status'] == 'skipped'])}")
print(f"Errors: {len(df_summary[df_summary['status'] == 'error'])}")
print(f"\nTotal new rows imported: {df_summary[df_summary['status'] == 'success']['rows'].sum()}")
print()
display(df_summary)

In [0]:
# 6. Ensure primary key constraints exist on all imported tables
pk_results = []

for row in results:
    if row["status"] != "success":
        continue
    
    table_name = row["target"]
    constraint_name = f"pk_{table_name}"
    
    try:
        spark.sql(f"ALTER TABLE {table_name} ALTER COLUMN row_id SET NOT NULL")
        spark.sql(f"ALTER TABLE {table_name} ADD CONSTRAINT {constraint_name} PRIMARY KEY (row_id)")
        
        print(f"  \u2713 {table_name}: PRIMARY KEY (row_id) added")
        pk_results.append({"table": table_name, "status": "success"})
    except Exception as e:
        error_msg = str(e)[:300]
        if "already_exist" in error_msg.lower() or "already exists" in error_msg.lower():
            print(f"  \u2713 {table_name}: PRIMARY KEY already exists")
            pk_results.append({"table": table_name, "status": "exists"})
        else:
            print(f"  \u2717 {table_name}: {error_msg}")
            pk_results.append({"table": table_name, "status": "error"})

print(f"\nPrimary keys ensured: {len([r for r in pk_results if r['status'] in ('success', 'exists')])} / {len(pk_results)}")